# Core rates in our own generations: OLMo 3 Instruct SFT vs DPO

**Data:** `new_stories/data/instruct-sft.jsonl` and `instruct-dpo.jsonl`, 500 stories each
(5 prompts × 100 samples), generated with `helper_scripts/generate_stories.py` on llama.cpp (Q8_0 GGUF).

**Question:** does the Core rate in what the checkpoints actually *write* jump between SFT and DPO?

**Two things to keep in mind:*
- These are the **Instruct** branch checkpoints, not Think
- The system prompt in these runs has an extra line, *"Each story should be about 500 words."*, which the
  project's standard system prompt (and the old 5,000-story OLMo Think run) doesn't have. Comparisons with the
  old run are therefore approximate; the SFT-vs-DPO comparison here is like-for-like.

## 0. Setup

Run this notebook with the project's `uv` tooling from the project root, e.g.

```bash
uv run --no-project --with jupyterlab,pandas jupyter lab
```

The scorer is imported from `helper_scripts/core_rates.py`, so the Core word list and whole-word matching rule are
**exactly** the ones verified against the paper's Table 5.

In [1]:
import json
import math
import sys
from collections import Counter
from pathlib import Path

import pandas as pd
from IPython.display import display

# Locate the project root (works whether the notebook is run from the root or from notebooks/).
ROOT = Path.cwd()
while not (ROOT / "helper_scripts" / "core_rates.py").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Run this notebook from inside the project folder.")
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "helper_scripts"))
from core_rates import CORE, PATTERN, WORDS, score_core  # noqa: E402  (the paper's scorer)

DATA = ROOT / "training_stories" / "data"  # Dolci training extracts, used for context in section 7

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)


def has_core_first_n(story: str, n: int) -> bool:
    # True if the story's first n words contain a Core word.
    return bool(PATTERN.search(" ".join(story.split()[:n])))


print("Project root:", ROOT)

Project root: /Users/mihirjoshi/AI Research - Cornell


## 1. Setup check: were both stages run the same way?

Project rule: same setup for every checkpoint. We check that both files agree on everything except the model.

In [2]:
NEW = ROOT / "new_stories" / "data"
STAGES = ["instruct-sft", "instruct-dpo"]
gen = pd.concat([pd.read_json(NEW / f"{s}.jsonl", lines=True, precise_float=True) for s in STAGES], ignore_index=True)
gen["stage"] = pd.Categorical(gen["stage"], STAGES, ordered=True)


def one_value(s: pd.Series) -> str:
    # The single value a column takes, or a warning if it takes several.
    vals = s.map(lambda v: json.dumps(v, sort_keys=True) if isinstance(v, dict) else str(v)).unique()
    return vals[0] if len(vals) == 1 else f"!! {len(vals)} different values"


checks = pd.DataFrame({
    stage: {
        "stories": len(g),
        "repo @ revision": f"{one_value(g['repo'])} @ {one_value(g['revision'])[:10]}",
        "llama.cpp build": one_value(g["llama_cpp_build"]),
        "chat template sha": one_value(g["chat_template_sha256"]),
        "sampling": one_value(g["sampling"]),
        "same system prompt as other stage": None,
        "stories per prompt": ", ".join(str(n) for n in g["prompt"].value_counts(sort=False)),
        "unique seeds": g["seed"].nunique(),
        "finish_reason": ", ".join(f"{k}: {v}" for k, v in g["finish_reason"].value_counts().items()),
        "non-empty reasoning": int((g["reasoning"].str.len() > 0).sum()),
        "story contains <think>": int(g["story"].str.contains("<think>|</think>").sum()),
        "story == raw output (trimmed)": int((g["story"].str.strip() == g["raw_content"].str.strip()).sum()),
        "story_words field correct": int((g["story_words"] == g["story"].str.split().str.len()).sum()),
    }
    for stage, g in gen.groupby("stage", observed=True)
})
same_sys = gen.groupby("stage", observed=True)["system_prompt"].agg(set)
checks.loc["same system prompt as other stage"] = same_sys.iloc[0] == same_sys.iloc[1]
print("System prompt used:\n" + gen["system_prompt"].iloc[0] + "\n")
checks

System prompt used:
You are a fiction writer.
When the user asks for a story, respond with only the story text. Do not ask clarifying questions. Do not add introductions, markdown separators, bullet lists, or follow-up suggestions.
Each story should be about 500 words.



,instruct-sft,instruct-dpo
stories,500,500
repo @ revision,allenai/Olmo-3-7B-Instruct-SFT @ e1452fc572,allenai/Olmo-3-7B-Instruct-DPO @ b33130b7de
llama.cpp build,b11633-5cdadfc32,b11633-5cdadfc32
chat template sha,f199ceca02ab,f199ceca02ab
sampling,"{""max_tokens"": 4096, ""min_p"": 0.05, ""temperature"": 0.6, ""top_k"": 40, ""top_p"": 0.95}","{""max_tokens"": 4096, ""min_p"": 0.05, ""temperature"": 0.6, ""top_k"": 40, ""top_p"": 0.95}"
same system prompt as other stage,True,True
stories per prompt,"100, 100, 100, 100, 100","100, 100, 100, 100, 100"
unique seeds,500,500
finish_reason,"stop: 499, length: 1",stop: 500
non-empty reasoning,0,0


one of the sft stories hit the 500 word limit

In [3]:
# The one truncated story (hit the 4,096-token cap): what happened?
cut = gen[gen["finish_reason"] != "stop"]
for _, row in cut.iterrows():
    words = row["story"].split()
    print(f"{row['stage']} | prompt {row['prompt']!r} | seed {row['seed']} | {len(words):,} words, "
          f"{row['usage']['completion_tokens']:,} tokens")
    print("  start:", " ".join(words[:40]))
    print("  end:  ", " ".join(words[-40:]))

instruct-sft | prompt 'Please tell a story.' | seed 400054 | 3,534 words, 4,024 tokens
  start: The city of Luminis was built on the ruins of something older, something that had once thrived with the same wild, untamed joy that the people of Luminis now sought in the flickering lights of their neon-lit streets. The old
  end:   of the city, a piece of the world, and she knew that she would never be able to give it back. But she also knew that she had stolen something that was worth it, worth any risk, worth any danger.


## 2. Run the scorer script

This calls `helper_scripts/core_rates.py` exactly as on the command line (`jsonl ... --field story`), so the
numbers are the verified Table 5 scorer's. The `story` field already has the reasoning removed (and these
Instruct models produce none), so no `--strip-thinking` is needed.

In [4]:
import subprocess

SCORER = ROOT / "helper_scripts" / "core_rates.py"


def run_scorer(paths, field):
    # Run core_rates.py on JSONL files and return its JSON output keyed by file stem.
    out = subprocess.run([sys.executable, str(SCORER), "jsonl", *map(str, paths), "--field", field],
                         capture_output=True, text=True, check=True).stdout
    return {Path(r["file"]).stem: r for r in json.loads(out)}


script = run_scorer([NEW / f"{s}.jsonl" for s in STAGES], "story")

summary = pd.DataFrame({
    s: {
        "stories": f"{r['records']:,}",
        "empty stories": f"{r['empty_records']:,}",
        "stories with ≥1 Core word": f"{r['records_with_core']:,}",
        "Core %": f"{r['percent_with_core']:.1f}%",
        "total words": f"{r['word_count']:,}",
        "Core PPM (hits per million words)": f"{r['core_ppm']:,.1f}",
    }
    for s, r in ((s, script[s]) for s in STAGES)
})
summary

,instruct-sft,instruct-dpo
stories,500,500
empty stories,0,0
stories with ≥1 Core word,222,267
Core %,44.4%,53.4%
total words,"265,047","306,981"
Core PPM (hits per million words),"6,270.6","6,042.7"


## 3. Is the SFT → DPO difference real?

95% Wilson confidence intervals for each Core %, and a two-sided Fisher exact test for the difference. The two
stages are independent samples (same seeds, but different models, so seed *i* in SFT and seed *i* in DPO aren't
related stories), so this is an unpaired test.

We also recompute the rate in the notebook with `score_core()` to confirm it agrees with the script, and check that
dropping the one truncated story doesn't change anything.

In [5]:
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]:
    # 95% Wilson score interval for a proportion, in percent.
    p = k / n
    d = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / d
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return 100 * (centre - half), 100 * (centre + half)


def fisher_exact(a: int, b: int, c: int, d: int) -> float:
    # Two-sided Fisher exact p-value for the 2x2 table [[a, b], [c, d]].
    n1, n2, k = a + b, c + d, a + c
    prob = lambda x: math.comb(n1, x) * math.comb(n2, k - x) / math.comb(n1 + n2, k)
    p0 = prob(a)
    return min(1.0, sum(prob(x) for x in range(max(0, k - n2), min(k, n1) + 1) if prob(x) <= p0 * (1 + 1e-9)))


scores = gen["story"].map(score_core)
gen["has_core"] = scores.map(lambda r: r["has_core"])
gen["words"] = scores.map(lambda r: r["word_count"])
gen["counts"] = scores.map(lambda r: r["counts"])
for group, words in CORE.items():
    gen[f"has_{group}"] = gen["counts"].map(lambda c, ws=words: any(c[w] for w in ws))


def compare(df: pd.DataFrame, col: str = "has_core") -> dict:
    # Core % per stage with CI, plus the Fisher test of SFT vs DPO.
    k = df.groupby("stage", observed=True)[col].sum().astype(int)
    n = df.groupby("stage", observed=True)[col].size()
    row = {}
    for s in STAGES:
        lo, hi = wilson(k[s], n[s])
        row[s] = f"{100 * k[s] / n[s]:.1f}%  [{lo:.1f}, {hi:.1f}]  ({k[s]}/{n[s]})"
    a, c = k[STAGES[0]], k[STAGES[1]]
    row["DPO − SFT (pp)"] = round(100 * (c / n[STAGES[1]] - a / n[STAGES[0]]), 1)
    row["Fisher p"] = f"{fisher_exact(a, n[STAGES[0]] - a, c, n[STAGES[1]] - c):.2g}"
    return row


assert (gen.groupby("stage", observed=True)["has_core"].sum().values
        == [script[s]["records_with_core"] for s in STAGES]).all(), "notebook and script disagree"
print("Notebook scoring matches the script.\n")

pd.DataFrame({
    "all stories": compare(gen),
    "finished stories only": compare(gen[gen["finish_reason"] == "stop"]),
    "has a Core name": compare(gen, "has_names"),
    "has a Core profession": compare(gen, "has_professions"),
    "has 'lighthouse'": compare(gen, "has_locations"),
}).T

Notebook scoring matches the script.



,instruct-sft,instruct-dpo,DPO − SFT (pp),Fisher p
all stories,"44.4% [40.1, 48.8] (222/500)","53.4% [49.0, 57.7] (267/500)",9.0,0.0054
finished stories only,"44.3% [40.0, 48.7] (221/499)","53.4% [49.0, 57.7] (267/500)",9.1,0.0044
has a Core name,"27.4% [23.7, 31.5] (137/500)","27.6% [23.9, 31.7] (138/500)",0.2,1
has a Core profession,"22.0% [18.6, 25.8] (110/500)","27.2% [23.5, 31.3] (136/500)",5.2,0.066
has 'lighthouse',"8.4% [6.3, 11.2] (42/500)","18.6% [15.4, 22.2] (93/500)",10.2,3e-06


## 4. Each prompt separately

The five prompts are near-paraphrases, so they should give similar rates. A big spread would mean the single-prompt
old run isn't representative.

In [6]:
by_prompt = (gen.groupby(["prompt", "stage"], observed=True, sort=False)["has_core"].mean().mul(100).round(1)
             .unstack("stage"))
by_prompt["DPO − SFT"] = (by_prompt[STAGES[1]] - by_prompt[STAGES[0]]).round(1)
by_prompt.loc["ALL PROMPTS"] = [round(100 * gen[gen["stage"] == s]["has_core"].mean(), 1) for s in STAGES] + [
    round(100 * (gen[gen["stage"] == STAGES[1]]["has_core"].mean() - gen[gen["stage"] == STAGES[0]]["has_core"].mean()), 1)]
print("Core % by prompt (100 stories per cell)")
by_prompt

Core % by prompt (100 stories per cell)


stage,instruct-sft,instruct-dpo,DPO − SFT
prompt,,,
Write a story.,45.0,50.0,5.0
Write me a story.,39.0,59.0,20.0
Please write a story.,39.0,42.0,3.0
Tell me a story.,55.0,63.0,8.0
Please tell a story.,44.0,53.0,9.0
ALL PROMPTS,44.4,53.4,9.0


## 5. Which Core words?

For each word: the share of stories that contain it at least once, and its frequency per million words (PPM).

In [7]:
rows = {}
for group, words in CORE.items():
    for w in words:
        row = {}
        for s in STAGES:
            r = script[s]["per_word"][w]
            row[(s, "% stories")] = round(100 * r["records_with_word"] / script[s]["records"], 1)
            row[(s, "PPM")] = round(r["ppm"], 1)
        rows[(w, group)] = row
per_word = pd.DataFrame.from_dict(rows, orient="index").rename_axis(["word", "group"])
per_word[("DPO / SFT", "PPM ratio")] = (per_word[(STAGES[1], "PPM")]
                                        / per_word[(STAGES[0], "PPM")].replace(0, float("nan"))).round(2)
per_word.sort_values((STAGES[1], "% stories"), ascending=False)

instruct-sft         instruct-dpo         DPO / SFT
                          % stories     PPM    % stories     PPM PPM ratio
word       group                                                          
elara      names               21.2  3652.2         21.4  3003.4      0.82
lighthouse locations            8.4  1056.4         18.6  1599.4      1.51
keeper     professions          6.0   192.4         12.6   309.5      1.61
baker      professions          6.4   166.0          6.0   107.5      0.65
mara       names                4.2   690.4          3.8   566.8      0.82
librarian  professions          7.6   162.2          3.2    58.6      0.36
fisherman  professions          0.6    11.3          3.0    65.2      5.77
clockmaker professions          0.8    30.2          2.6    78.2      2.59
elias      names                2.2   222.6          2.4   218.3      0.98
conductor  professions          0.6    30.2          2.0    35.8      1.19
mayor      professions          1.2    56.6          0.0     0.0      0.00

In [8]:
# How many of the three Core groups (names / professions / lighthouse) does each story hit?
cats = gen["counts"].map(lambda c: sum(any(c[w] for w in ws) for ws in CORE.values()))
(pd.crosstab(cats, gen["stage"], normalize="columns").mul(100).round(1)
 .rename(index=lambda i: f"{i} of 3 groups").rename_axis("Core groups hit").rename_axis(None, axis=1)
 .astype(str) + "%")

,instruct-sft,instruct-dpo
Core groups hit,,
0 of 3 groups,55.6%,46.6%
1 of 3 groups,32.8%,36.6%
2 of 3 groups,9.8%,13.6%
3 of 3 groups,1.8%,3.2%


## 6. Length control

DPO outputs could simply be longer, which gives more chances to hit a Core word
. Two checks: Core PPM (already in section 2) and the Core rate in **only the first 300 words** of every story.

In [8]:
FIRST_N = 300
gen["has_core_first_n"] = [has_core_first_n(s, FIRST_N) for s in gen["story"]]

length = gen.groupby("stage", observed=True)["words"].describe(percentiles=[0.1, 0.5, 0.9])[
    ["mean", "10%", "50%", "90%", "max"]].round(0).astype(int)
length.columns = ["mean words", "p10", "median", "p90", "max"]
length["share < 300 words"] = gen.groupby("stage", observed=True)["words"].apply(lambda w: f"{100 * (w < FIRST_N).mean():.1f}%")
display(length)
pd.DataFrame({"Core %, whole story": compare(gen), f"Core %, first {FIRST_N} words": compare(gen, "has_core_first_n")}).T

,mean words,p10,median,p90,max,share < 300 words
stage,,,,,,
instruct-sft,530,411,524,644,3534,0.6%
instruct-dpo,614,493,604,742,1015,0.0%


,instruct-sft,instruct-dpo,DPO − SFT (pp),Fisher p
"Core %, whole story","44.4% [40.1, 48.8] (222/500)","53.4% [49.0, 57.7] (267/500)",9.0,0.0054
"Core %, first 300 words","42.4% [38.1, 46.8] (212/500)","49.0% [44.6, 53.4] (245/500)",6.6,0.042


## 7. Context: other models and the training data

Training-data rates for the **Instruct** branch come from running the same script on the `dolci-instruct-*` extracts
(`--field text`). The Think-branch training rates are the paper's Table 2. The other models are the existing
5,000-story runs in `core_rates.json` (single prompt "write a story", old system prompt).

In [9]:
train = run_scorer([DATA / f"dolci-instruct-{s}.jsonl" for s in ("sft", "dpo", "rl")], "text")
old = {Path(r["file"]).stem: r for r in json.loads((ROOT / "core_rates.json").read_text())}
label = {"anthropic-claude-haiku-4.5": "Claude Haiku 4.5", "google-gemini-3.1-flash-lite": "Gemini 3.1 Flash Lite",
         "openai-gpt-5.4-mini": "GPT-5.4 mini", "local-Olmo-3-7B-Think": "OLMo 3 7B Think (final, RLVR)"}


def row(kind, name, r):
    return {"kind": kind, "what": name, "records": r["records"], "Core %": round(r["percent_with_core"], 1),
            "Core PPM": round(r["core_ppm"], 1)}


context = [row("our generations", f"OLMo 3 7B Instruct-{s.split('-')[1].upper()}", script[s]) for s in STAGES]
context += [row("Instruct training data", f"dolci-instruct-{s}", train[f"dolci-instruct-{s}"]) for s in ("sft", "dpo", "rl")]
context += [{"kind": "Think training data (paper Table 2)", "what": f"Think {s}", "records": None, "Core %": v, "Core PPM": None}
            for s, v in (("SFT", 3.0), ("DPO", 8.0), ("RL", 10.9))]
for stem, r in old.items():
    name = next((v for k, v in label.items() if k in stem), stem)
    context.append(row("existing 5,000-story runs", name, r))
context = pd.DataFrame(context).set_index(["kind", "what"])
context["records"] = context["records"].map(lambda v: "" if pd.isna(v) else f"{v:,.0f}")
context["Core PPM"] = context["Core PPM"].map(lambda v: "" if pd.isna(v) else f"{v:,.1f}")
context

records  Core %  Core PPM
kind                                what                                                   
our generations                     OLMo 3 7B Instruct-SFT            500    44.4   6,270.6
                                    OLMo 3 7B Instruct-DPO            500    53.4   6,042.7
Instruct training data              dolci-instruct-sft             57,846     2.3     183.3
                                    dolci-instruct-dpo              5,011     8.2     881.1
                                    dolci-instruct-rl               1,976    10.7     372.8
Think training data (paper Table 2) Think SFT                                 3.0          
                                    Think DPO                                 8.0          
                                    Think RL                                 10.9          
existing 5,000-story runs           Claude Haiku 4.5                5,000    91.4  13,438.7
                                    Gemini 3.1 Flash Lite           5,000    99.0  16,887.0
                                    GPT-5.4 mini                    5,000    89.6  14,946.8
                                    OLMo 3 7B Think (final, RLVR)   5,000    73.1  11,034.6

## 8. Beyond the 11 Core words: which names and places recur?

The Core list was built from *other* models' favourites. A checkpoint can be just as repetitive with different
names. Here we count capitalized words that appear **mid-sentence** (so ordinary sentence-starting words are mostly
excluded) and report the share of stories that use each. Crude, but it surfaces character names and place names.

In [11]:
import re

SENT_SPLIT = re.compile(r"(?<=[.!?…])[\"”’)]*\s+")
CAP = re.compile(r"^[\"“‘(]*([A-Z][a-z]{2,})[^A-Za-z]*$")
NOT_NAMES = set('''The She He Her His They Their It Its But And When Then There This That What Who Why How Where
Not For With From Into One Some Each Every All Even Yet Still Just Only Now Once Maybe Perhaps Mother Father Mom Dad
Grandma Grandpa God Lord Sir Miss Mrs Mr Dr You Your Our We Thank Yes Okay Please Good Hello Come Let Don Can Sunday
Monday Tuesday Wednesday Thursday Friday Saturday January February March April May June July August September
October November December English Christmas'''.split())


def mid_sentence_caps(story: str) -> set[str]:
    # Capitalized words that are not the first word of a sentence (mostly names and places).
    found = set()
    for sent in SENT_SPLIT.split(story):
        for tok in sent.split()[1:]:
            m = CAP.match(tok)
            if m and m.group(1) not in NOT_NAMES:
                found.add(m.group(1))
    return found


gen["caps"] = gen["story"].map(mid_sentence_caps)
core_set = {w.capitalize() for w in WORDS}
tops = {}
for s in STAGES:
    df = Counter(w for caps in gen.loc[gen["stage"] == s, "caps"] for w in caps)
    n = (gen["stage"] == s).sum()
    tops[s] = [f"{w}{'*' if w in core_set else ''}  {100 * c / n:.0f}%" for w, c in df.most_common(20)]
print("Top 20 mid-sentence capitalized words, % of stories using each (* = on the Core list)")
pd.DataFrame(tops, index=range(1, 21))

Top 20 mid-sentence capitalized words, % of stories using each (* = on the Core list)


,instruct-sft,instruct-dpo
1,Elara* 21%,Clara 22%
2,Lila 18%,Elara* 20%
3,Mira 12%,End 11%
4,Clara 11%,Mira 11%
5,Willowbrook 7%,Lila 9%
6,Maya 6%,Eli 6%
7,Willow 5%,Cove 6%
8,Eli 4%,Street 5%
9,Mara* 4%,Eleanor 5%
10,Maple 3%,Maya 5%


In [12]:
# How concentrated are the protagonists? Take the story's most frequent mid-sentence capitalized word as a proxy.
def top_name(story: str) -> str | None:
    c = Counter()
    for sent in SENT_SPLIT.split(story):
        for tok in sent.split():
            m = CAP.match(tok)
            if m and m.group(1) not in NOT_NAMES:
                c[m.group(1)] += 1
    return c.most_common(1)[0][0] if c else None


gen["protagonist"] = gen["story"].map(top_name)
conc = {}
for s in STAGES:
    p = gen.loc[gen["stage"] == s, "protagonist"].dropna()
    vc = p.value_counts()
    conc[s] = {
        "distinct protagonist names": p.nunique(),
        "top-1 name share": f"{vc.index[0]} {100 * vc.iloc[0] / len(p):.1f}%",
        "top-5 names share": f"{100 * vc.iloc[:5].sum() / len(p):.1f}%",
        "top-5 names": ", ".join(vc.index[:5]),
    }
pd.DataFrame(conc)

,instruct-sft,instruct-dpo
distinct protagonist names,86,72
top-1 name share,Elara 19.5%,Clara 23.4%
top-5 names share,60.0%,56.8%
top-5 names,"Elara, Lila, Clara, Mira, Maya","Clara, Elara, Mira, Lila, Eleanor"


In [13]:
# Openings: the most common first four words of a story.
gen["opening"] = gen["story"].map(lambda t: " ".join(t.split()[:4]))
op = {}
for s in STAGES:
    vc = gen.loc[gen["stage"] == s, "opening"].value_counts()
    op[s] = [f"{o}  ({c})" for o, c in vc.head(8).items()]
    op[s] += [""] * (8 - len(op[s]))
    print(f"{s}: {gen.loc[gen['stage'] == s, 'opening'].nunique()} distinct 4-word openings out of 500")
pd.DataFrame(op, index=range(1, 9))

instruct-sft: 193 distinct 4-word openings out of 500
instruct-dpo: 168 distinct 4-word openings out of 500


,instruct-sft,instruct-dpo
1,On the edge of (50),On the edge of (65)
2,In the heart of (47),The old lighthouse stood (40)
3,The old lighthouse stood (27),The rain fell in (21)
4,The rain fell in (15),Rain spattered against the (18)
5,In the quiet town (14),In the quiet town (16)
6,In the city of (13),The clock above the (14)
7,"Once upon a time, (13)",The city lights shimmered (13)
8,The sky above the (10),The rain came down (11)


In [14]:
# Recurring place names. "End" in the table above is mostly "Marrow’s End", a town the DPO model keeps reusing.
PLACE = re.compile(r"\b(?:[A-Z][a-z]+[’']s (?:End|Hollow|Rest|Reach|Landing|Point)"
                   r"|[A-Z][a-z]+ (?:Hollow|Cove|Bay|Harbor|Creek|Falls|Haven)"
                   r"|[A-Z][a-z]+(?:brook|wood|haven|ville|shire|ford|mere))\b")
places = {}
for s in STAGES:
    df = Counter(p for story in gen.loc[gen["stage"] == s, "story"] for p in set(PLACE.findall(story)))
    places[s] = [f"{p}  ({c})" for p, c in df.most_common(10)]
print("Most common place names (number of stories, out of 500)")
pd.DataFrame(places, index=range(1, 11))

Most common place names (number of stories, out of 500)


,instruct-sft,instruct-dpo
1,Willowbrook (35),Marrow’s End (56)
2,Willow Creek (16),Silverwood (10)
3,Silver Bay (4),Saltwater Cove (10)
4,Marrow’s End (3),Saltmarsh Cove (8)
5,New Haven (2),Silverbrook (4)
6,Silverbrook (2),Silver Bay (4)
7,Maplewood (2),Willow Creek (4)
8,Willow’s End (2),Silver Cove (4)
9,Driftwood (2),Willowbrook (4)
10,Elden Hollow (1),New Haven (3)


## 9. Scorer check: read the Core hits in context

Whole-word matching can still be fooled (e.g. *conductor* as electricity, *mayor* as a surname). Show one
short context window for a random sample of hits of each word.

In [15]:
rng_state = 0
for w in WORDS:
    hits = []
    for s, story in zip(gen["stage"], gen["story"]):
        for m in re.finditer(rf"\b{w}\b", story, re.IGNORECASE):
            hits.append((s, " ".join(story[max(0, m.start() - 70): m.end() + 50].split())))
    if not hits:
        continue
    print(f"== {w}: {len(hits)} hits")
    for s, ctx in pd.Series(hits).sample(min(3, len(hits)), random_state=rng_state):
        print(f"   [{s}] …{ctx}…")

== elias: 126 hits
   [instruct-sft] …on lights flickered against midnight shadows, there lived a man named Elias. He was a quiet soul, often lost in the labyrinth…
   [instruct-sft] …in the dim light. “I believe that even shadows can have a purpose.” Elias smiled, his face softening. “Then perhaps you wil…
   [instruct-sft] …e music guide you. Let me show you.” Together, they spent the night. Elias’s music filled her studio, and Aria’s brushes mov…


== mara: 357 hits
   [instruct-sft] …ns flickered, casting long shadows that seemed to beckon her forward. Mara smiled, a little wistfully, a little bravely. The…
   [instruct-sft] …rinned, a flash of white teeth in the gloom. He held out his paw, and Mara saw that his claws were clean. She hesitated, the…
   [instruct-dpo] …e, something shifted—a quietening, perhaps, or a softening. Finally, Mara asked, “And what happens when we find the answers…
== elara: 1890 hits
   [instruct-sft] …ne’s lamp swayed, casting the lighthouse in a golden glow. “And when Elara reached the shore, she found no one waiting. The…
   [instruct-dpo] …s. Veyse sat rigid in her armchair, her hands folded in her lap. When Elara revealed the canvas, the room seemed to breathe.…
   [instruct-sft] …forgotten. Tonight, as the moon hung low and silvered the rooftops, Elara found herself drawn toward the city’s oldest libr…
== keeper: 146 hits
   [instruct-sft] …here stood an old lighthouse. Its lantern had not shone 

== mayor: 15 hits
   [instruct-sft] …to contain stories older than Willowbrook itself. A hush fell as the mayor, a wiry man with a voice like rustling leaves, li…
   [instruct-sft] …every ending is just the start of a new story.” Mara watched as the mayor raised a lantern, its flame bright and steady. Th…
   [instruct-sft] …eeling in the grass, her lantern crushed in her hands. “Help us,” the mayor pleaded. “The magic is out of control. The Clockt…
== clockmaker: 32 hits
   [instruct-dpo] …k, and opened it to a blank page. “Once,” she murmured, “there was a clockmaker who could mend not only broken timepieces but the…
   [instruct-dpo] …e neon shimmered like a second sky, there was a small shop called The Clockmaker’s Nook. Its sign flickered in the rain, spelling…
   [instruct-dpo] …uite kept, the fear that she’d never be enough. As she spoke, the old clockmaker listened, his hands moving deftly over the gears,…


== fisherman: 23 hits
   [instruct-dpo] …here everyone believed the storm was a curse. But one night, a lonely fisherman found a lantern floating on the river, its light…
   [instruct-dpo] …mmer, Clara had been lost for hours on the cliffs, found at last by a fisherman who swore he’d seen her silhouette against the gl…
   [instruct-dpo] …their movements: the shy boy who hurried home to his mother, the old fisherman casting his net, the woman walking her cat throug…


== librarian: 61 hits
   [instruct-sft] …rgy. Later that week, Elara took the stone to the village’s wise old librarian, an elderly woman named Mira. Mira examined the s…
   [instruct-sft] …sprawling city, there lived a young woman named Elara who worked as a librarian. Her days were filled with the quiet magic of boo…
   [instruct-dpo] …s glances. Maribel, however, was not a superstitious soul. She was a librarian by trade, and libraries had always been sanctuari…


== conductor: 19 hits
   [instruct-dpo] …choolbook too heavy for his back. Mrs. Liao moved between them like a conductor, her voice a soft melody that soothed even the mo…
   [instruct-sft] …e old train rattled awake beneath a sky the color of fresh cream. The conductor, a man with a voice like gravel and a cap pulled…
   [instruct-dpo] …but polite reserve kept her thoughts to herself. A bell jingled. The conductor—a thin woman with a clipboard and a permanent fro…
== lighthouse: 771 hits
   [instruct-dpo] …The old lighthouse stood sentinel at the edge of Greymere Bay, its w…
   [instruct-dpo] …t across the horizon, painting gold over the mist, Maribel turned the lighthouse back on. Its beam pierced the morning, steady and…
   [instruct-sft] …even hear his voice carried on the wind. That night, she climbed the lighthouse again, her heart pounding as she pressed her hand…


## 10. Interpretation

*(Written 2026-10-26 from the outputs above; re-check if the data or scorer changes.)*

**Setup: clean.** Both stages used the same llama.cpp build, chat template, system prompt, sampling
(T 0.6, top_p 0.95, top_k 40, min_p 0.05, 4,096-token cap) and 100 stories per prompt. No reasoning, no `<think>`
tags. One SFT story hit the token cap (a 3,534-word story); dropping it changes nothing.

**Headline: Core rate rises from 44.4% (SFT) to 53.4% (DPO), +9.0 points, Fisher p ≈ 0.005.** The 95% intervals
are about ±4.4 points at n = 500, so the gap is real but moderate. It shows up for all five prompts (+3 to +20).

**The rise is almost all *lighthouse* and *keeper*, not names.**
- Stories with a Core **name**: 27.4% → 27.6% (no change). *Elara* stays at about 21% of stories at both stages.
- Stories with **lighthouse**: 8.4% → 18.6% (more than double, p ≈ 3e-6); **keeper** 6.0% → 12.6%.
  The 4-word opening "The old lighthouse stood" goes from 27 to 40 stories.
- This matches an earlier check of the Think DPO training pairs, where *lighthouse* (≈6×) and *keeper* (≈3.5×)
  were the words most over-represented on the chosen side. (Caveat: that was Think DPO data; these are
  Instruct checkpoints trained on `dolci-instruct-dpo`.)

**Length explains part of it, not all.** DPO stories are ~15% longer (median 604 vs 524 words). Looking only at
the first 300 words, the gap shrinks to 42.4% → 49.0% (+6.6, p ≈ 0.04) but survives. Overall Core PPM is flat
(6,271 → 6,043) because *Elara* (by far the most frequent Core word) drops slightly per word while *lighthouse* rises
about 1.5× per word.

**Most of the amplification happens before DPO.** The Instruct SFT training data is only 2.3% Core, yet the
Instruct-SFT model already writes Core stories 44% of the time, roughly 19× the data rate. DPO data is 8.2% Core
and the DPO model adds 9 more points. So for the Instruct branch the big jump is at **SFT** (or earlier, in
pre-/mid-training), with DPO adding a smaller, word-specific push. Whether the Think branch looks the same (final
Think model 73.1%) is the main open question.

**DPO narrows things beyond the Core list too.**
- Distinct protagonist names: 86 → 72; distinct 4-word openings: 193 → 168 (of 500).
- **Clara** (not a Core word) becomes the top name in DPO (22% of stories, up from 11%). Elara/Clara/Mira/Lila
  together cover most protagonists at both stages.
- **"Marrow’s End"** appears in 56 DPO stories vs 3 SFT stories: a new favourite setting the Core list can't see.
  The Core list was built from other models' habits, so it undercounts OLMo-specific repetition.

**Scorer check.** Sampled hits read as intended (character names, lighthouse keepers, bakers). Edge cases:
"Lantern Keeper" counts as *keeper*, and "moved like a conductor" counts as *conductor*. These are rare and don't
change the conclusions.

## 11. Caveats and next steps
- **Instruct, not Think.** The research question is about the Think branch. Run the same analysis on Think
  SFT / DPO / RLVR.
- **System prompt differs** from the project standard (extra "about 500 words" line). For the Think runs, use the
  exact system prompt in `CLAUDE.md` (or ask the mentor which to standardize on), so results compare with the old
  73.1% run.
- **Missing the last Instruct stage.** Adding `Olmo-3-7B-Instruct` (after RL) would complete this branch's curve.
- **Score the Instruct DPO pairs** (chosen vs rejected) to see whether *lighthouse*/*keeper*
  are over-represented on the chosen side there too.
- Consider extending the scorer with OLMo-specific favourites (Clara, Lila, Mira, Marrow’s End) as a secondary metric.